# Solar Filament CV V2: budgeted nested grouped evaluation
## Gregory Kojadinovich II

**Experimental, not a scored submission. V1 and its active Kaggle run are unchanged.**
Train grayscale U-Net/ResNet18 from fresh ImageNet initialization in every trial.
Compare 512/1024-pixel input and raw/CLAHE preprocessing, then two seeded random
hyperparameter candidates. Three outer folds are grouped by observing day; each
outer training partition has a separate day-grouped inner validation holdout.
This is **outer grouped K-fold with inner holdout tuning**, not inner K-fold CV.
All annotator targets stay with their physical image.

The default wall-time allowance is **two hours for nested training and scoring**,
checked between batches/images. An individual operation, installation, model
downloads, and optional test export can exceed/add to that allowance.
18 training trials (6 configurations x 3 folds), up to 60 epochs each, will
likely NOT converge or finish within two hours. Budget truncation is an
experimental outcome, not convergence. Increase `hours` with a **new run_name**
for longer training. Use a fresh Kaggle output directory, T4 GPU, and Internet.
No globally optimal parameters or numerical gains are claimed.

Hypotheses: higher resolution may preserve thin structures; CLAHE may expose
faint filaments but amplify artifacts; longer training may reduce underfitting
but also overfit. Compare measured PQ/Dice/IoU and fold standard deviations.
The previously viewed V1 holdout is not a new untouched final test for V2.

In [ ]:
%pip install -q 'segmentation-models-pytorch>=0.3.3,<0.7' pycocotools scipy 'opencv-python-headless>=4.9,<5'

In [ ]:
import csv
import hashlib
import importlib.metadata
import itertools
import json
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
from PIL import Image
import pycocotools.mask as mask_utils
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CFG = dict(seed=2026, image_size=512, batch_size=8, epochs=20,
           learning_rate=3e-4, encoder='resnet18', architecture='Unet',
           pretrained=True, workers=0, tune_fraction=0.15, holdout_fraction=0.15,
           tune_image_limit=32, thresholds=[0.35, 0.5, 0.65],
           min_areas=[64, 256], contrast_thresholds=[0.025, 0.04, 0.06])
RUN_NAME = 'v2_data_setup'
RUN_TRAINING = False
RUN_CALIBRATION = False
SMOKE_TEST = False
SUBMIT_TO_KAGGLE = False
DATA_ROOT_OVERRIDE = None
CHECKPOINT_INPUT_DIR = None  # Path to an attached experiment folder for inference-only reproduction.
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
              else Path.cwd() / 'cv_outputs') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path(CHECKPOINT_INPUT_DIR) if CHECKPOINT_INPUT_DIR else OUTPUT_DIR
CHECKPOINT_PATH = INPUT_DIR / 'best_model.pt'
COMPETITION = 'filament-segmentation-2026'
DATASET_NAME = 'MAGFiLO_1.0_Kaggle_2026'
ANNOTATION_NAME = 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
FULL_SHAPE = (2048, 2048)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    print('WARNING: CPU selected. Enable a Kaggle GPU before full training.')
if CFG['image_size'] % 32 or CFG['batch_size'] < 2 or CFG['epochs'] < 1:
    raise ValueError('Use image_size divisible by 32, batch_size >= 2, and epochs >= 1.')
print('Device:', DEVICE, 'outputs:', OUTPUT_DIR)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False))


## Budget and search settings
Change `V2` **before** running helpers. `max_epochs=60` is a ceiling, not a
requirement to reach epoch 60; early stopping uses inner loss (patience 8,
minimum significant decrease 1e-4). Strictly lower inner loss still saves
the best checkpoint. ReduceLROnPlateau halves the learning rate after three
non-improving epochs.

Four controlled ablations share LR=3e-4, AdamW weight decay=0.01, batch=2,
and CLAHE clip limit=2/grid=8x8 where enabled. Two additional 1024+CLAHE trials
sample LR log-uniformly from 1e-5 to 1e-3, weight decay 1e-5 to 1e-2, batch
2 or 4, and clip limit 1.5/2/3. These are bounded samples, not an exhaustive
or optimal search. Batch sizes differ from V1 for high-resolution memory.
The 512/raw control is retrained in nested folds; it is not the frozen V1 score.
No automatic OOM fallback changes experimental parameters: use a new plan
with a smaller batch if your accelerator cannot fit a candidate.

CLAHE is applied **after resize**, identically in training/validation/test.
All inputs remain single-channel, normalized to [-1,1].

In [ ]:
import gc
import cv2

V2 = dict(
    hours=2.0, folds=3, max_epochs=60, patience=8, min_delta=1e-4,
    inner_fraction=0.2, random_trials=2, seed=2026,
    calibration_images=12, thresholds=[0.35, 0.5, 0.65], min_areas=[64, 256],
    run_name="v2_nested_1024_clahe_seed2026",
)
RUN_V2 = True
RUN_TEST_EXPORT = False
# Each rerun gets this wall-time allowance, reusing completed immutable trials.
# Increase hours and use a NEW run_name to allocate longer training per trial.
V2_DIR = OUTPUT_DIR.parent / V2["run_name"]
V2_DIR.mkdir(parents=True, exist_ok=True)




In [ ]:
candidates = ([Path(DATA_ROOT_OVERRIDE)] if DATA_ROOT_OVERRIDE else [
    Path('/kaggle/input/competitions') / COMPETITION / DATASET_NAME,
    Path('/kaggle/input') / COMPETITION / DATASET_NAME,
    Path.cwd() / 'data' / DATASET_NAME, Path.cwd() / DATASET_NAME])
if DATA_ROOT_OVERRIDE is None and Path('/kaggle/input').is_dir():
    candidates += sorted(Path('/kaggle/input').glob(f'*/{DATASET_NAME}'))
    candidates += sorted(Path('/kaggle/input').glob(f'*/*/{DATASET_NAME}'))
valid_roots = [p for p in candidates if (p / 'train' / ANNOTATION_NAME).is_file()
               and (p / 'test' / 'test_images').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Attach the competition data or set DATA_ROOT_OVERRIDE to its dataset root.')
DATA_ROOT = valid_roots[0]
train_paths = sorted((DATA_ROOT / 'train' / 'train_images').glob('*.jpeg'))
test_paths = sorted((DATA_ROOT / 'test' / 'test_images').glob('*.jpeg'))
if not train_paths or not test_paths:
    raise FileNotFoundError('Train and test must both contain JPEG files.')
with (DATA_ROOT / 'train' / ANNOTATION_NAME).open() as stream:
    coco = json.load(stream)
records = coco['images']
if len({r['id'] for r in records}) != len(records):
    raise ValueError('Duplicate COCO image IDs.')
records_by_name = defaultdict(list)
annotations_by_id = defaultdict(list)
for record in records:
    if (record['height'], record['width']) != FULL_SHAPE:
        raise ValueError(f'Unexpected dimensions for {record["id"]}')
    records_by_name[record['file_name']].append(record)
for ann in coco['annotations']:
    if ann.get('iscrowd', 0) != 0:
        raise ValueError('Local PQ assumes individual non-crowd filaments.')
    annotations_by_id[ann['image_id']].append(ann)
known_ids = {r['id'] for r in records}
if set(annotations_by_id) - known_ids:
    raise ValueError('Orphan annotation image IDs.')
if len({a['id'] for a in coco['annotations']}) != len(coco['annotations']):
    raise ValueError('Duplicate filament annotation IDs.')
train_lookup = {p.name: p for p in train_paths}
if set(records_by_name) - set(train_lookup):
    raise FileNotFoundError('Annotations reference missing training files.')
if set(train_lookup) & {p.name for p in test_paths}:
    raise ValueError('Train/test filenames overlap.')

annotation_sha256 = hashlib.sha256((DATA_ROOT / 'train' / ANNOTATION_NAME).read_bytes()).hexdigest()
print('V2 data:', len(train_paths), 'labeled JPEGs;', len(test_paths), 'test JPEGs')


In [ ]:
@lru_cache(maxsize=128)
def small_image(filename):
    with Image.open(train_lookup[filename]) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{filename}: expected 2048 x 2048 grayscale JPEG.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.float32) / 255.0

def ann_rle(annotation):
    segmentation = annotation['segmentation']
    if isinstance(segmentation, list):
        if not segmentation or any(len(p) < 6 or len(p) % 2 for p in segmentation):
            raise ValueError(f'Invalid polygon: {annotation["id"]}')
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, *FULL_SHAPE))
    elif isinstance(segmentation, dict):
        if list(segmentation['size']) != list(FULL_SHAPE):
            raise ValueError('RLE dimensions do not match the image.')
        if isinstance(segmentation['counts'], list):
            rle = mask_utils.frPyObjects(segmentation, *FULL_SHAPE)
        else:
            counts = segmentation['counts']
            rle = dict(size=list(FULL_SHAPE), counts=counts.encode('ascii') if isinstance(counts, str) else counts)
    else:
        raise TypeError(f'Unsupported segmentation: {annotation["id"]}')
    if mask_utils.area(rle) <= 0:
        raise ValueError(f'Empty ground-truth filament: {annotation["id"]}')
    return rle

@lru_cache(maxsize=512)
def gt_rles(image_id):
    return [ann_rle(a) for a in annotations_by_id[image_id]]

def union_mask(rles):
    if not rles:
        return np.zeros(FULL_SHAPE, dtype=np.uint8)
    return mask_utils.decode(mask_utils.merge(rles))

@lru_cache(maxsize=128)
def small_target(image_id):
    full_mask = union_mask(gt_rles(image_id))
    resized = Image.fromarray(full_mask).resize((CFG['image_size'], CFG['image_size']), Image.Resampling.NEAREST)
    return np.asarray(resized, dtype=np.float32)

class FilamentDataset(Dataset):
    def __init__(self, entries, augment=False):
        self.entries = entries
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        record = self.entries[index]
        image = small_image(record['file_name'])
        target = small_target(record['id'])
        if self.augment:
            turns = random.randrange(4)
            image, target = np.rot90(image, turns), np.rot90(target, turns)
            if random.random() < 0.5:
                image, target = np.fliplr(image), np.fliplr(target)
        x = torch.from_numpy(np.ascontiguousarray((image - 0.5) / 0.5))[None]
        y = torch.from_numpy(np.ascontiguousarray(target))[None]
        return x, y



In [ ]:
def build_model(config, pretrained=False):
    constructors = {'Unet': smp.Unet, 'FPN': smp.FPN}
    if config['architecture'] not in constructors:
        raise ValueError('Supported architectures: Unet, FPN.')
    return constructors[config['architecture']](encoder_name=config['encoder'],
        encoder_weights='imagenet' if pretrained else None, in_channels=1, classes=1, activation=None)

def segmentation_loss(logits, target):
    probabilities = logits.sigmoid()
    intersection = (probabilities * target).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = 1 - ((2 * intersection + 1) / (denominator + 1)).mean()
    return 0.5 * F.binary_cross_entropy_with_logits(logits, target) + 0.5 * dice



In [ ]:
@torch.inference_mode()
def predict_probability(path):
    with Image.open(path) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{path.name}: invalid image mode or dimensions.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np.ascontiguousarray((array - 0.5) / 0.5))[None, None].to(DEVICE)
    probability = model(tensor).sigmoid()[0, 0].cpu().numpy()
    if not np.isfinite(probability).all():
        raise FloatingPointError(f'Nonfinite prediction: {path.name}')
    return probability

def full_probability(probability):
    resized = Image.fromarray(np.asarray(probability, dtype=np.float32)).resize(
        (2048, 2048), Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32)

def contrast_foreground(filename, threshold):
    image = small_image(filename)
    disk = ndi.binary_erosion(image > 0.08, iterations=max(1, CFG['image_size']//128))
    background = ndi.gaussian_filter(image, sigma=CFG['image_size']/32)
    return disk & ((background - image) > threshold)

def instance_rles(probability, threshold, min_area):
    if min_area < 1 or not 0 <= threshold <= 1:
        raise ValueError('Invalid threshold or minimum area.')
    binary = full_probability(probability) >= threshold
    labels, count = ndi.label(binary, structure=np.ones((3, 3), dtype=np.uint8))
    areas = np.bincount(labels.ravel())
    result = []
    for label in range(1, count+1):
        if areas[label] >= min_area:
            mask = np.asfortranarray(labels == label, dtype=np.uint8)
            result.append(mask_utils.encode(mask))
    return result

def local_scores(predicted, truth):
    n_pred, n_true = len(predicted), len(truth)
    overlaps = (mask_utils.iou(predicted, truth, [0]*n_true)
                if n_pred and n_true else np.zeros((n_pred, n_true)))
    matched = []
    if n_pred and n_true:
        valid = overlaps > 0.5
        # Maximum cardinality first, then overlap; greedy matching can lose true positives.
        rewards = valid * (1 + overlaps / (min(n_pred, n_true) + 1))
        rows, columns = linear_sum_assignment(-rewards)
        matched = [float(overlaps[r, c]) for r, c in zip(rows, columns) if valid[r, c]]
    tp = len(matched); fp = n_pred-tp; fn = n_true-tp
    denominator = tp + 0.5*fp + 0.5*fn
    pq = sum(matched)/denominator if denominator else 1.0
    pred_union, true_union = union_mask(predicted).astype(bool), union_mask(truth).astype(bool)
    intersection = np.count_nonzero(pred_union & true_union)
    union = np.count_nonzero(pred_union | true_union)
    pixels = np.count_nonzero(pred_union) + np.count_nonzero(true_union)
    return dict(pq=pq, dice=2*intersection/pixels if pixels else 1.0,
                iou=intersection/union if union else 1.0, tp=tp, fp=fp, fn=fn,
                split_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=0) > 1)),
                merge_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=1) > 1)))

def evaluate(names, parameters, probability_cache=None):
    rows = []
    for filename in names:
        if parameters['method'] == 'unet':
            probability = (probability_cache[filename] if probability_cache is not None
                           else predict_probability(train_lookup[filename]))
            threshold = parameters['threshold']
        elif parameters['method'] == 'contrast':
            probability = contrast_foreground(filename, parameters['contrast_threshold']).astype(np.float32)
            threshold = 0.5
        else:
            raise ValueError('Unknown segmentation method.')
        predicted = instance_rles(probability, threshold, parameters['min_area'])
        for record in records_by_name[filename]:
            rows.append(dict(filename=filename, image_id=record['id'],
                             **local_scores(predicted, gt_rles(record['id']))))
    if not rows:
        raise ValueError('Evaluation set is empty.')
    # Average annotators per image, then physical images; do not overweight triply labeled images.
    grouped = defaultdict(list)
    for row in rows: grouped[row['filename']].append(row)
    metrics = {key: float(np.mean([np.mean([r[key] for r in group]) for group in grouped.values()]))
               for key in ['pq', 'dice', 'iou', 'tp', 'fp', 'fn', 'split_relations', 'merge_relations']}
    return metrics, rows


In [ ]:
class BudgetExpired(RuntimeError):
    pass


def check_deadline(deadline):
    if time.monotonic() >= deadline:
        raise BudgetExpired("Wall-time budget reached at a safe compute boundary.")


def v2_candidates(settings):
    base = dict(learning_rate=3e-4, weight_decay=0.01, batch_size=2, clip_limit=2.0)
    candidates = [
        dict(name=name, image_size=size, clahe=clahe, **base)
        for name, size, clahe in [
            ("raw512", 512, False), ("clahe512", 512, True),
            ("raw1024", 1024, False), ("clahe1024", 1024, True),
        ]
    ]
    rng = np.random.default_rng(settings["seed"])
    for index in range(settings["random_trials"]):
        candidates.append(dict(
            name=f"search{index+1}", image_size=1024, clahe=True,
            learning_rate=float(10 ** rng.uniform(-5, -3)),
            weight_decay=float(10 ** rng.uniform(-5, -2)),
            batch_size=int(rng.choice([2, 4])),
            clip_limit=float(rng.choice([1.5, 2.0, 3.0])),
        ))
    return candidates


def grouped_folds(names, settings):
    days = sorted({name[:8] for name in names})
    random.Random(settings["seed"]).shuffle(days)
    if len(days) < settings["folds"] * 3:
        raise ValueError("Too few observation days for nested grouped splits.")
    outer_days = [set(chunk.tolist()) for chunk in np.array_split(days, settings["folds"])]
    folds = []
    for index, outer in enumerate(outer_days):
        remaining = sorted(set(days) - outer)
        random.Random(settings["seed"] + index).shuffle(remaining)
        count = max(1, round(len(remaining) * settings["inner_fraction"]))
        inner = set(remaining[:count])
        training = set(remaining[count:])
        parts = dict(train=training, inner=inner, outer=outer)
        assert all(not parts[a] & parts[b] for a, b in itertools.combinations(parts, 2))
        fold = {key: sorted(n for n in names if n[:8] in group) for key, group in parts.items()}
        if not all(fold.values()):
            raise ValueError("Empty nested fold.")
        folds.append(fold)
    assert sorted(n for fold in folds for n in fold["outer"]) == sorted(names)
    return folds


def preprocess_v2(path, config):
    with Image.open(path) as image:
        if image.mode != "L" or image.size != (2048, 2048):
            raise ValueError(f"{path.name}: expected full-resolution grayscale observation.")
        array = np.asarray(image.resize(
            (config["image_size"], config["image_size"]), Image.Resampling.BILINEAR
        ))
    if config["clahe"]:
        array = cv2.createCLAHE(
            clipLimit=config["clip_limit"], tileGridSize=(8, 8)
        ).apply(array)
    return np.ascontiguousarray(array, dtype=np.float32) / 255.0


class V2Dataset(Dataset):
    def __init__(self, names, config, augment=False):
        self.entries = [record for name in names for record in records_by_name[name]]
        self.config, self.augment = config, augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        record = self.entries[index]
        image = preprocess_v2(train_lookup[record["file_name"]], self.config)
        mask = Image.fromarray(union_mask(gt_rles(record["id"]))).resize(
            (self.config["image_size"], self.config["image_size"]), Image.Resampling.NEAREST
        )
        target = np.asarray(mask, dtype=np.float32)
        if self.augment:
            turns = random.randrange(4)
            image, target = np.rot90(image, turns), np.rot90(target, turns)
            if random.random() < 0.5:
                image, target = np.fliplr(image), np.fliplr(target)
        return (
            torch.from_numpy(np.ascontiguousarray((image - 0.5) / 0.5))[None],
            torch.from_numpy(np.ascontiguousarray(target))[None],
        )


@torch.inference_mode()
def v2_probability(network, path, config):
    array = preprocess_v2(path, config)
    tensor = torch.from_numpy((array - 0.5) / 0.5)[None, None].to(DEVICE)
    probability = network(tensor).sigmoid()[0, 0].cpu().numpy()
    if not np.isfinite(probability).all():
        raise FloatingPointError(f"Nonfinite V2 prediction: {path.name}")
    return probability


def new_v2_model(pretrained):
    return smp.Unet(encoder_name="resnet18",
                    encoder_weights="imagenet" if pretrained else None,
                    in_channels=1, classes=1, activation=None).to(DEVICE)


def atomic_checkpoint(path, payload):
    temporary = path.with_suffix(".tmp.pt")
    torch.save(payload, temporary)
    temporary.replace(path)


def train_trial(config, fold, directory, deadline, max_seconds):
    directory.mkdir(parents=True, exist_ok=True)
    seed = V2["seed"]
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    network = new_v2_model(CFG["pretrained"])
    optimizer = torch.optim.AdamW(
        network.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )
    loaders = {
        key: DataLoader(V2Dataset(fold[key], config, augment=key == "train"),
            batch_size=config["batch_size"], shuffle=key == "train", num_workers=0,
            drop_last=key == "train" and
            len(V2Dataset(fold[key], config)) % config["batch_size"] == 1)
        for key in ["train", "inner"]
    }
    if any(len(loader) == 0 for loader in loaders.values()):
        raise ValueError("No complete training or validation batches.")
    history, best, significant_best, stale = [], float("inf"), float("inf"), 0
    trial_end = min(deadline, time.monotonic() + max_seconds)
    status = "max_epochs"
    try:
        for epoch in range(V2["max_epochs"]):
            totals = {}
            for phase, loader in loaders.items():
                network.train(phase == "train")
                total, count = 0.0, 0
                with torch.set_grad_enabled(phase == "train"):
                    for images, targets in loader:
                        check_deadline(trial_end)
                        images, targets = images.to(DEVICE), targets.to(DEVICE)
                        optimizer.zero_grad(set_to_none=True)
                        loss = segmentation_loss(network(images), targets)
                        if not torch.isfinite(loss):
                            raise FloatingPointError("Nonfinite V2 loss.")
                        if phase == "train":
                            loss.backward()
                            optimizer.step()
                        total += loss.item() * len(images); count += len(images)
                totals[phase] = total / count
            scheduler.step(totals["inner"])
            history.append(dict(epoch=epoch+1, **totals,
                                learning_rate=optimizer.param_groups[0]["lr"]))
            print(config["name"], history[-1], flush=True)
            if totals["inner"] < best:
                best = totals["inner"]
                atomic_checkpoint(directory / "best_model.pt", dict(
                    state_dict={k: v.detach().cpu() for k, v in network.state_dict().items()},
                    config=config, epoch=epoch+1, inner_loss=best,
                    annotation_sha256=annotation_sha256, smoke_test=SMOKE_TEST,
                ))
            if totals["inner"] < significant_best - V2["min_delta"]:
                significant_best, stale = totals["inner"], 0
            else:
                stale += 1
            write_json(directory / "history.json", history)
            if stale >= V2["patience"]:
                status = "early_stopped"
                break
    except BudgetExpired as error:
        print("BUDGET:", config["name"], str(error), flush=True)
        status = "budget_truncated"
    finally:
        del network, optimizer, scheduler, loaders
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return dict(status=status, epochs=len(history), history=history)


def load_v2_model(path):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if saved["annotation_sha256"] != annotation_sha256 or saved["smoke_test"] != SMOKE_TEST:
        raise ValueError("V2 checkpoint provenance mismatch.")
    network = new_v2_model(False)
    network.load_state_dict(saved["state_dict"], strict=True)
    return network.eval(), saved["config"]


def score_v2(network, names, config, parameters, deadline, cache=None):
    rows = []
    for name in names:
        check_deadline(deadline)
        probability = cache[name] if cache is not None else v2_probability(network, train_lookup[name], config)
        predicted = instance_rles(probability, parameters["threshold"], parameters["min_area"])
        annotators = [local_scores(predicted, gt_rles(r["id"])) for r in records_by_name[name]]
        rows.append(dict(filename=name, **{
            key: float(np.mean([scores[key] for scores in annotators]))
            for key in annotators[0]
        }))
    if not rows:
        raise ValueError("V2 evaluation set is empty.")
    return {key: float(np.mean([r[key] for r in rows]))
            for key in rows[0] if key != "filename"}, rows


def calibrate_v2(path, names, deadline):
    network, config = load_v2_model(path)
    names = list(names)
    random.Random(V2["seed"]).shuffle(names)
    names = names[:V2["calibration_images"]]
    cache = {}
    for name in names:
        check_deadline(deadline)
        cache[name] = v2_probability(network, train_lookup[name], config)
    results = []
    for threshold, area in itertools.product(V2["thresholds"], V2["min_areas"]):
        parameters = dict(threshold=threshold, min_area=area)
        metrics, _ = score_v2(network, names, config, parameters, deadline, cache)
        results.append(dict(parameters=parameters, metrics=metrics))
    del network, cache
    return dict(
        **max(results, key=lambda row: row["metrics"]["pq"]),
        filenames=names, candidates=results,
        checkpoint_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
    )


def run_nested_v2(candidates=None, folds=None):
    candidates = v2_candidates(V2) if candidates is None else candidates
    folds = grouped_folds(sorted(records_by_name), V2) if folds is None else folds
    plan = dict(settings=V2, candidates=candidates, folds=folds,
                annotation_sha256=annotation_sha256, smoke_test=SMOKE_TEST,
                pretrained=CFG["pretrained"], torch_version=str(torch.__version__),
                opencv_version=cv2.__version__)
    plan_path = V2_DIR / "plan.json"
    if plan_path.exists():
        if json.loads(plan_path.read_text()) != plan:
            raise ValueError("Existing V2 plan differs. Choose a new run_name/output directory.")
    else:
        write_json(plan_path, plan)
    if not np.isfinite(V2["hours"]) or V2["hours"] <= 0:
        raise ValueError("V2 hours must be finite and positive.")
    deadline = time.monotonic() + V2["hours"] * 3600
    jobs = [(i, config) for i in range(len(folds)) for config in candidates]
    results = []
    try:
        for job_index, (fold_index, config) in enumerate(jobs):
            check_deadline(deadline)
            directory = V2_DIR / f"fold{fold_index+1}" / config["name"]
            result_path = directory / "result.json"
            if result_path.exists():
                result = json.loads(result_path.read_text())
                checkpoint = directory / "best_model.pt"
                if result.get("status") == "evaluated":
                    if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != result["calibration"]["checkpoint_sha256"]:
                        raise ValueError("Completed trial checkpoint changed.")
                    results.append(result)
                    continue
            # Share the remaining wall budget; reserve time for scoring and outer evaluation.
            share = max(0, deadline - time.monotonic()) / (len(jobs) - job_index)
            training_path = directory / "training_status.json"
            if training_path.is_file():
                training = json.loads(training_path.read_text())
            else:
                # Never reuse an interrupted, uncommitted training checkpoint.
                checkpoint = directory / "best_model.pt"
                if checkpoint.exists():
                    checkpoint.unlink()
                training = train_trial(config, folds[fold_index], directory, deadline, share * 0.55)
                write_json(training_path, training)
            checkpoint = directory / "best_model.pt"
            result = dict(fold=fold_index+1, name=config["name"], training=training,
                          status="no_complete_epoch")
            if checkpoint.is_file():
                calibration = calibrate_v2(checkpoint, folds[fold_index]["inner"], deadline)
                result.update(status="evaluated", calibration=calibration)
            write_json(result_path, result)
            results.append(result)
            write_json(V2_DIR / "trial_results.json", results)
        outer_results = []
        selections = []
        for index, fold in enumerate(folds):
            eligible = [r for r in results if r["fold"] == index+1 and r["status"] == "evaluated"]
            if len(eligible) != len(candidates):
                print("INCOMPLETE: not every candidate has inner evaluation for fold", index+1)
                continue
            best = max(eligible, key=lambda r: r["calibration"]["metrics"]["pq"])
            # Choose using INNER scores only; outer labels cannot affect selection.
            selections.append(dict(fold=index+1, name=best["name"]))
            for result in eligible:
                check_deadline(deadline)
                path = V2_DIR / f"fold{index+1}" / result["name"] / "best_model.pt"
                network, config = load_v2_model(path)
                metrics, rows = score_v2(
                    network, fold["outer"], config, result["calibration"]["parameters"], deadline
                )
                outer_results.append(dict(
                    fold=index+1, name=result["name"], selected=result["name"] == best["name"],
                    metrics=metrics, training_status=result["training"]["status"],
                ))
                write_json(path.parent / "outer_predictions_metrics.json", rows)
                write_json(V2_DIR / "outer_results.json", outer_results)
                del network
        selected = [r for r in outer_results if r["selected"]]
        complete = len(selected) == len(folds)
        summary = dict(
            complete=complete, completed_outer_folds=len(selected), planned_outer_folds=len(folds),
            selections=selections, outer_results=outer_results,
            note="Budget-limited nested selection; not a globally optimal hyperparameter search.",
        )
        if complete:
            summary["nested_metrics"] = {
                key: dict(mean=float(np.mean([r["metrics"][key] for r in selected])),
                          std=float(np.std([r["metrics"][key] for r in selected], ddof=1)))
                for key in ["pq", "dice", "iou"]
            }
        summary["ablation_metrics"] = {}
        for config in candidates:
            measured = [r for r in outer_results if r["name"] == config["name"]]
            if len(measured) == len(folds):
                summary["ablation_metrics"][config["name"]] = {
                    key: dict(mean=float(np.mean([r["metrics"][key] for r in measured])),
                              std=float(np.std([r["metrics"][key] for r in measured], ddof=1)))
                    for key in ["pq", "dice", "iou"]
                }
        write_json(V2_DIR / "summary.json", summary)
        print(json.dumps(summary, indent=2))
    except BudgetExpired as error:
        print("INCOMPLETE RUN:", error, "Completed artifacts are preserved in", V2_DIR)
        write_json(V2_DIR / "budget_stop.json", dict(
            status="incomplete", message=str(error), evaluated_trials=len(results),
            planned_trials=len(jobs),
        ))
        return None
    return summary


def export_v2_test():
    summary = json.loads((V2_DIR / "summary.json").read_text())
    if SMOKE_TEST or not summary["complete"]:
        print("NOT READY: complete non-smoke nested evaluation is required before test export.")
        return None
    chosen = summary["selections"]
    # One row-generating prediction per test image from averaged fold probabilities.
    # Select a common threshold using INNER calibrations only, never outer/test scores.
    models, thresholds, areas = [], [], []
    for selection in chosen:
        directory = V2_DIR / f'fold{selection["fold"]}' / selection["name"]
        result = json.loads((directory / "result.json").read_text())
        path = directory / "best_model.pt"
        if hashlib.sha256(path.read_bytes()).hexdigest() != result["calibration"]["checkpoint_sha256"]:
            raise ValueError("Selected V2 checkpoint changed.")
        models.append((path, result))
        thresholds.append(result["calibration"]["parameters"]["threshold"])
        areas.append(result["calibration"]["parameters"]["min_area"])
    parameters = dict(threshold=float(np.median(thresholds)), min_area=int(np.median(areas)))
    output = V2_DIR / "submission.csv"
    temporary = output.with_suffix(".tmp.csv")
    audit, row_count = [], 0
    with temporary.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["filament_id", "segmentation_rle"])
        writer.writeheader()
        for path in tqdm(test_paths, desc="V2 test ensemble"):
            probability = np.zeros(FULL_SHAPE, dtype=np.float32)
            for checkpoint_path, _ in models:
                network, config = load_v2_model(checkpoint_path)
                probability += full_probability(v2_probability(network, path, config)) / len(models)
                del network
            instances = instance_rles(probability, **parameters)
            audit.append(dict(filename=path.name, instances=len(instances)))
            for index, rle in enumerate(instances, 1):
                counts = rle["counts"].decode("ascii")
                mask = mask_utils.decode(rle)
                if mask.shape != FULL_SHAPE or not mask.any():
                    raise ValueError("Invalid V2 instance mask.")
                if mask_utils.encode(np.asfortranarray(mask))["counts"].decode("ascii") != counts:
                    raise ValueError("V2 COCO RLE roundtrip failed.")
                writer.writerow(dict(filament_id=f"{path.stem}_{index}", segmentation_rle=counts))
                row_count += 1
    if row_count == 0:
        temporary.unlink()
        print("NOT READY: no predictions. Audit:", audit)
        return None
    with temporary.open(newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["filament_id", "segmentation_rle"] or sum(1 for _ in reader) != row_count:
            raise ValueError("V2 CSV readback failed.")
    temporary.replace(output)
    write_json(V2_DIR / "test_manifest.json", dict(
        parameters=parameters, selections=chosen, images=audit, rows=row_count,
        csv_sha256=hashlib.sha256(output.read_bytes()).hexdigest(),
        checkpoint_sha256=[r["calibration"]["checkpoint_sha256"] for _, r in models],
        warning="Ensemble threshold is an inner-calibration median heuristic, not separately optimized.",
    ))
    zeros = [r["filename"] for r in audit if r["instances"] == 0]
    if zeros:
        print("WARNING: zero-detection image rows omitted; Kaggle acceptance unverified:", zeros)
    if Path("/kaggle/working").is_dir():
        shutil.copyfile(output, "/kaggle/working/submission.csv")
    print("V2 CSV saved:", output, "No API upload performed.")
    return output


## Run the nested experiment
Every outer fold is excluded from all training, inner checkpoint selection,
and post-processing calibration for that fold. Candidate selection uses inner
PQ on up to 12 **physical images**, averaging annotators within each image.
The same seeded inner subset is used across candidates. Calibration considers
thresholds 0.35/0.5/0.65 and minimum original-resolution areas 64/256.
All outer images are evaluated only after selection; no test labels are used.

The remaining wall allowance is shared across unprocessed jobs, with 55% of
each share assigned to training and the rest reserved for scoring. Allocation
is hardware-dependent, not a fixed-epoch comparison; report executed epochs.
Incomplete epochs are discarded. `budget_truncated` does not mean early-stop
convergence. A trial requires at least one full train+validation epoch before
its checkpoint can be calibrated. A missing candidate prevents a complete
fold-selection report. Mean/std is emitted only for all three completed folds.

Artifacts are written after every complete epoch/trial and outer evaluation.
Rerunning the **same unchanged plan** reuses completed trials and can finish
pending scoring; it does not extend previously budget-truncated training.
Do not delete downloaded checkpoints. For a larger training budget, use a new
run name. Unexpected errors remain visible; only explicit budget expiration
is converted into a clearly marked incomplete run.

In [ ]:
if RUN_V2:
    v2_summary = run_nested_v2()
else:
    print("V2 training is disabled. Saved artifacts remain available:", V2_DIR)


## Measured convergence and reproducibility
Curves show only completed epochs, not interpolated convergence. Save the
entire V2 output folder, especially `plan.json`, fold checkpoint/result files,
histories, `summary.json`, and actual package versions. A missing summary or
`complete=False` is not a completed three-fold comparison.

In [ ]:
paths = sorted(V2_DIR.glob("fold*/*/history.json"))
if paths:
    figure, axis = plt.subplots(figsize=(12, 5))
    for path in paths:
        history = json.loads(path.read_text())
        axis.plot([row["epoch"] for row in history], [row["inner"] for row in history],
                  label=f"{path.parent.parent.name}/{path.parent.name}")
    axis.set(xlabel="Completed epoch", ylabel="Inner validation Dice+BCE loss")
    axis.legend(fontsize=6, ncol=3)
    figure.tight_layout()
    figure.savefig(V2_DIR / "inner_loss_curves.png", dpi=150)
    plt.show()
with (V2_DIR / "requirements-actual.txt").open("w") as stream:
    subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                   stdout=stream, check=True)
print("Evidence directory:", V2_DIR)


## Optional test export (off by default)
Enable `RUN_TEST_EXPORT` only after a complete non-smoke nested evaluation.
The three inner-selected fold models are ensembled by averaging **full-resolution
probabilities**, then applying the median of their inner-calibrated thresholds
and minimum areas. This median is a declared heuristic, not a calibrated
ensemble optimum. Outer scores do not select models or post-processing.
Model loading per image bounds GPU memory but adds overhead.

Export performs no API upload. Zero-detection images are audited without
fabricating masks; Kaggle acceptance of omitted rows is still unverified.
Nested outer scores evaluate the selection procedure's individual fold models,
**not** the final three-model ensemble on an independent labeled test set.
Do not call those fold metrics an ensemble score.
Download the experiment folder before resetting a session.

In [ ]:
if RUN_TEST_EXPORT:
    v2_submission_path = export_v2_test()
else:
    print("Test export disabled; V1 submission is unaffected.")


## References and reporting checklist
- Ronneberger et al., U-Net (2015), doi:10.1007/978-3-319-24574-4_28.
- He et al., Deep Residual Learning (2016), doi:10.1109/CVPR.2016.90.
- Kirillov et al., Panoptic Segmentation (2019), doi:10.1109/CVPR.2019.00963.
- Bergstra & Bengio, [Random Search for Hyper-Parameter Optimization](https://jmlr.org/papers/v13/bergstra12a.html), JMLR 13, 281-305 (2012).
- Varma & Simon, [Bias in error estimation when using cross-validation for model selection](https://doi.org/10.1186/1471-2105-7-91), BMC Bioinformatics 7, 91 (2006).
- OpenCV, [Histogram Equalization / CLAHE](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html).

Report actual epochs, stopping reason, runtime/hardware, fold membership,
searched parameters, calibration subset, PQ/Dice/IoU mean and sample SD,
failure/incomplete-fold counts, zero predictions, and genuine Kaggle scores.
No numerical expected gains are inserted as results. A continued decline
in training loss alone does not demonstrate improved generalization.
The organizers' final form additionally requires the report and reproducible
public repository; a Kaggle score alone is not the full deliverable.